# Classification III: probabilities and choosing a model
## The model says 0.48. What do you do with that?
**Salem Othman · Student notebook**

Last class ended with a hard fact: on the refund data, a model that answers "no" to everybody is right 92% of the time and catches nobody, and the tree that caught 15 of 16 refund requests cost the gym four times less.

Today you get the tool that makes that choice adjustable: a model that returns a **probability** instead of a label. You decide where to cut it.

Then you compare four classifiers on the same split and defend one choice. That is exactly what Assignment 2 asks for.

### Today's five steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | Probability | Fit logistic regression and read its risk scores | E1 |
| 2 | Threshold | Move the cut and watch the cost move | E2 |
| 3 | Evidence | See Naive Bayes: prior + evidence → posterior | demo |
| 4 | Choice | Compare four classifiers on the same split | E3, E4 |
| 5 | Audit | Answer two AI claims with your own numbers | written |

**For every exercise:** predict → write the code → check → explain.

Keep the variable names exactly as written; the Check cells look for them.

In [16]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
RANDOM_STATE = 42
COST_MISSED = 40  # dollars lost when we fail to flag a refund request
COST_CALL = 2  # dollars of staff time per call


def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)


def ready(value, step):
    """True when the exercise above is done. Otherwise print a reminder and return False.

    This never raises, so a blank exercise cell can never stop the class with a
    red traceback: the check cell just tells the student what to do next.
    """
    if value is None:
        print(f"{step} is not finished yet.")
        print(
            "Write your code in the exercise cell above, run that cell, then run this one again."
        )
        return False
    return True


print("Setup complete.")

Setup complete.


# Part 0 · The refund data, again
Same 800 members, same split, same costs as last class: a missed refund request costs the gym \$40, an unnecessary call costs \$2.

In [17]:
refund_rng = np.random.default_rng(5)
n_refund = 800
refund_requested = refund_rng.binomial(1, 0.08, n_refund)

refunds = pd.DataFrame(
    {
        "member_id": [f"R{i:03d}" for i in range(1, n_refund + 1)],
        "visits_per_month": np.where(
            refund_requested == 1,
            np.clip(refund_rng.normal(5, 3, n_refund), 0, None),
            np.clip(refund_rng.normal(8.5, 4, n_refund), 0, None),
        )
        .round()
        .astype(int),
        "complaints": np.where(
            refund_requested == 1,
            refund_rng.poisson(1.2, n_refund),
            refund_rng.poisson(0.55, n_refund),
        ),
        "days_since_signup": np.where(
            refund_requested == 1,
            refund_rng.integers(10, 200, n_refund),
            refund_rng.integers(15, 400, n_refund),
        ),
        "refund_requested": refund_requested,
    }
)

feature_cols = ["visits_per_month", "complaints", "days_since_signup"]
X, y = refunds[feature_cols], refunds["refund_requested"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train), "with", int(y_train.sum()), "refund requests")
print("Test rows:", len(X_test), "with", int(y_test.sum()), "refund requests")

Training rows: 600 with 47 refund requests
Test rows: 200 with 16 refund requests


# Part 1 · A model that answers with a probability
So far every model answered yes or no. Logistic regression answers **how likely**, and that is a different kind of useful.

It works in two steps:
1. **A weighted score.** Multiply each feature by a weight the model learned, add them up, add a constant. More complaints push the score up; more visits push it down.
2. **Squash it into a probability.** The sigmoid function turns any score into a number between 0 and 1. A score of 0 becomes 0.50, a very negative score approaches 0, a very positive score approaches 1.

Despite the name "regression", the output is used for classification: we get a probability, then we decide what to do with it.

`predict()` gives a label using a cut at 0.50. `predict_proba()` gives the probability itself. Today we work with the probability.

### E1 · Fit it and read the risk scores
**Predict:** refund requests are 8% of the members. Do you expect many members to have a risk above 0.50?
No, there are few members to have a risk above 0.5. 

In [18]:
logreg = None
risk = None
risk_table = None


# Write your code below. Your code must create the variables set to None above.

# 1. logreg: a pipeline of StandardScaler() and LogisticRegression(random_state=RANDOM_STATE),
#    fitted on the training rows.
#    Hint: make_pipeline(...).fit(X_train, y_train)
logreg = make_pipeline(
    StandardScaler(), LogisticRegression(random_state=RANDOM_STATE)
).fit(X_train, y_train)

# 2. risk: the predicted probability of a refund request for each TEST member.
#    Hint: logreg.predict_proba(X_test)[:, 1]  (column 1 is the probability of class 1)
risk = logreg.predict_proba(X_test)[:, 1]

# 3. risk_table: a DataFrame with member_id, the three features, risk, and the true answer,
#    sorted from the highest risk down. Display the first ten rows.
#    Hint: members are refunds.loc[X_test.index, "member_id"]
risk_table = (
    refunds.loc[X_test.index]
    .assign(risk=risk.round(3))
    .assign(actually_asked=y_test)
    .sort_values(by="risk", ascending=False)
)
display(risk_table.head(10))

# 4. Print the smallest and the largest risk value.
print(f"Smallest risk value: {round(risk.min(), 3)}")
print(f"Largest risk value: {round(risk.max(), 3)}")

,member_id,visits_per_month,complaints,days_since_signup,refund_requested,risk,actually_asked
619,R620,8,3,25,1,0.534,1
4,R005,5,2,29,0,0.518,0
113,R114,5,2,50,0,0.461,0
761,R762,7,2,13,1,0.449,1
49,R050,0,0,43,1,0.413,1
169,R170,8,2,20,0,0.376,0
423,R424,12,3,16,0,0.339,0
567,R568,7,2,57,0,0.336,0
55,R056,5,1,35,1,0.334,1
615,R616,3,4,272,0,0.327,0


Smallest risk value: 0.001
Largest risk value: 0.534


**Check E1**

In [19]:
if ready(logreg, "E1"):
    check(
        "logreg is a pipeline with a scaler and logistic regression",
        len(logreg.steps) == 2,
    )
    check(
        "it was fitted on the training rows only",
        np.allclose(logreg.steps[0][1].mean_, X_train.mean().values),
    )
    check("risk holds one probability per test member", len(risk) == len(y_test))
    check(
        "every risk is a probability between 0 and 1",
        risk.min() >= 0 and risk.max() <= 1,
    )
    check(
        "risk_table is sorted from the highest risk down",
        risk_table["risk"].is_monotonic_decreasing,
    )
    check("the highest risk in the data is below 0.60", risk.max() < 0.60)

PASS  logreg is a pipeline with a scaler and logistic regression
PASS  it was fitted on the training rows only
PASS  risk holds one probability per test member
PASS  every risk is a probability between 0 and 1
PASS  risk_table is sorted from the highest risk down
PASS  the highest risk in the data is below 0.60


In [20]:
if ready(logreg, "E1"):
    weights = pd.Series(
        logreg.named_steps["logisticregression"].coef_[0], index=feature_cols
    ).round(2)
    print("What the model learned (on scaled features):")
    display(weights)
    default_pred = logreg.predict(X_test)
    print("With the default cut at 0.50:")
    print(
        confusion_matrix(y_test, default_pred),
        " caught",
        int(confusion_matrix(y_test, default_pred)[1, 1]),
        "of",
        int(y_test.sum()),
        "refund requests",
    )

What the model learned (on scaled features):


visits_per_month    -0.92
complaints           0.55
days_since_signup   -1.21
dtype: float64

With the default cut at 0.50:
[[183   1]
 [ 15   1]]  caught 1 of 16 refund requests


**Write:** look at the three weights and at the default 0.50 result. What is the model saying about members, and how many refund requests does the default cut actually catch?

*Your answer:*
The model identifies which features affect risk the most. 

# Part 2 · 0.50 is a decision, not a law
The model gives a probability. **You** choose where to cut it, and that choice is a business decision:

- **Lower the cut** (say 0.10): flag more members. You catch more refund requests and make more unnecessary calls. Recall up, precision down.
- **Raise the cut** (say 0.50): flag almost nobody. Precision up, recall down, and the members you wanted to find walk out.

Neither is "correct". The costs decide: \$40 per missed request, \$2 per call.

### E2 · Move the cut and price each one

In [21]:
threshold_table = None
best_threshold = None

thresholds = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]

# Write your code below. Your code must create the variables set to None above.

# 1. For every t in thresholds, label a member as "flag" when risk >= t, then collect:
#       threshold, flagged (FP + TP), caught (TP), missed (FN), precision, recall, cost
#    cost = missed * COST_MISSED + flagged * COST_CALL
#    Hint: pred = (risk >= t).astype(int)
#    Hint: tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
#    Hint: precision_score(y_test, pred, zero_division=0)
rows = []
for t in thresholds:
    pred = (risk >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    rows.append(
        {
            "threshold": t,
            "flagged": fp + tp,
            "caught": tp,
            "missed": fn,
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "cost": fn * COST_MISSED + (fp + tp) * COST_CALL,
        }
    )

# 2. threshold_table: those rows as a DataFrame. Display it.
threshold_table = pd.DataFrame(rows)
threshold_table.reset_index()
display(threshold_table)
# 3. best_threshold: the threshold with the LOWEST cost.
#    Hint: threshold_table.loc[threshold_table["cost"].idxmin(), "threshold"]
best_threshold = threshold_table.loc[threshold_table["cost"].idxmin(), "threshold"]
print(f"Best threshhold: {best_threshold}")

,threshold,flagged,caught,missed,precision,recall,cost
0,0.05,86,15,1,0.174419,0.9375,212
1,0.08,65,13,3,0.200000,0.8125,250
2,0.10,51,13,3,0.254902,0.8125,222
3,0.15,36,12,4,0.333333,0.7500,232
4,0.20,23,8,8,0.347826,0.5000,366
5,0.30,11,4,12,0.363636,0.2500,502
6,0.50,2,1,15,0.500000,0.0625,604


Best threshhold: 0.05


**Check E2**

In [22]:
if ready(threshold_table, "E2"):
    check(
        "threshold_table has one row per threshold",
        len(threshold_table) == len(thresholds),
    )
    check(
        "it reports flagged, caught, missed, precision, recall, and cost",
        set(threshold_table.columns)
        >= {"threshold", "flagged", "caught", "missed", "precision", "recall", "cost"},
    )
    check(
        "lowering the cut flags more members",
        threshold_table.sort_values("threshold")["flagged"].is_monotonic_decreasing,
    )
    check(
        "lowering the cut raises recall",
        threshold_table.sort_values("threshold")["recall"].is_monotonic_decreasing,
    )
    check(
        "the default 0.50 row catches almost nothing",
        threshold_table.loc[threshold_table["threshold"] == 0.50, "caught"].iloc[0]
        <= 2,
    )
    check(
        "best_threshold is the cheapest row, and it is well below 0.50",
        best_threshold
        == threshold_table.loc[threshold_table["cost"].idxmin(), "threshold"]
        and best_threshold < 0.3,
    )

PASS  threshold_table has one row per threshold
PASS  it reports flagged, caught, missed, precision, recall, and cost
PASS  lowering the cut flags more members
PASS  lowering the cut raises recall
PASS  the default 0.50 row catches almost nothing
PASS  best_threshold is the cheapest row, and it is well below 0.50


**Write:** which threshold would you put in front of the gym manager? Give the cost, and name one thing that is not in the cost column but matters.

*Your answer:* I would give them a threshold of 0.05, it would have a cost of 212. It only missed 1 request in total and had an accuracy of 93%. 

# Part 3 · Naive Bayes: prior plus evidence
A different way to reach a probability. Instead of learning weights, Naive Bayes counts.

Start with what you believe before seeing any features, the **prior**: 8 of every 100 members ask for a refund. Then let each feature move that belief, and you get the **posterior**.

**A worked example, by hand.** Of 100 members, 8 ask for a refund. Say 5 of those 8 had 2 or more complaints, and among the 92 who did not ask, 12 had 2 or more complaints.

In [23]:
asked_with_complaints = 5  # of the 8 who asked for a refund
not_asked_with_complaints = 12  # of the 92 who did not

prior = 8 / 100
posterior = asked_with_complaints / (asked_with_complaints + not_asked_with_complaints)
print(
    "Before we know anything, the chance this member asks for a refund:",
    round(prior, 3),
)
print("Once we know they have 2 or more complaints:", round(posterior, 3))

Before we know anything, the chance this member asks for a refund: 0.08
Once we know they have 2 or more complaints: 0.294


Two or more complaints moves the belief from 0.08 to about 0.29. That is the whole idea: **prior + evidence → posterior**.

**The "naive" part.** With several features, Naive Bayes assumes they add evidence independently, as if complaints told you nothing about visits. That is usually false, and the model works well anyway, which is why it survives.

**Where it shines:** lots of features, each weak, especially word counts. Spam filters are the classic case, and it comes back in the text-mining unit. `GaussianNB` suits numeric features like ours; `MultinomialNB` suits counts like words.

In [24]:
naive_bayes = make_pipeline(StandardScaler(), GaussianNB()).fit(X_train, y_train)
nb_pred = naive_bayes.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, nb_pred).ravel()
print(
    "Naive Bayes at its default cut: caught",
    tp,
    "of",
    int(y_test.sum()),
    "| flagged",
    fp + tp,
    "| cost $",
    fn * COST_MISSED + (fp + tp) * COST_CALL,
)

Naive Bayes at its default cut: caught 2 of 16 | flagged 6 | cost $ 572


# Part 4 · Which classifier do we hand over?
Four models, one split, one table. The question is not "which is best" in general; there is no such thing. The question is which one does **this** job at the lowest cost, and whether you can explain it to the manager.

### E3 · The comparison table

In [25]:
comparison = None

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=21)).fit(
    X_train, y_train
)
tree = DecisionTreeClassifier(
    max_depth=3, class_weight="balanced", random_state=RANDOM_STATE
).fit(X_train, y_train)

models = [
    ("knn", knn, None),
    ("tree (balanced)", tree, None),
    ("logistic regression", logreg, best_threshold),
    ("naive bayes", naive_bayes, None),
]

# Write your code below. Your code must create the variable set to None above.

# 1. For each model in models, predict on the test rows and collect:
#       model, accuracy, precision, recall, f1, missed (FN), calls (FP + TP), cost
#    For logistic regression use your best_threshold instead of predict():
#       pred = (logreg.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)
#    For the other three use model.predict(X_test).
#    Hint: models is a list of (name, model, threshold) triples; threshold is None
#    for the models that do not need one.
rows = []
for name, model, threshold in models:
    if threshold is not None:
        pred = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
    else:
        pred = model.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    missed = fn
    calls = fp + tp
    cost = missed * COST_MISSED + calls * COST_CALL

    rows.append(
        {
            "model": name,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "missed": missed,
            "calls": calls,
            "cost": cost,
        }
    )

# 2. comparison: those rows as a DataFrame, indexed by model name. Display it.
comparison = pd.DataFrame(rows).set_index("model")
display(comparison)

,accuracy,precision,recall,f1,missed,calls,cost
model,,,,,,,
knn,0.92,0.000000,0.0000,0.000000,16,0,640
tree (balanced),0.81,0.270833,0.8125,0.406250,3,48,216
logistic regression,0.64,0.174419,0.9375,0.294118,1,86,212
naive bayes,0.91,0.333333,0.1250,0.181818,14,6,572


**Check E3**

In [26]:
if ready(comparison, "E3"):
    check("comparison holds all four models", len(comparison) == len(models))
    check(
        "it reports accuracy, precision, recall, f1, missed, calls, and cost",
        set(comparison.columns)
        >= {"accuracy", "precision", "recall", "f1", "missed", "calls", "cost"},
    )
    check(
        "logistic regression used your threshold, not the default 0.50",
        comparison.loc["logistic regression", "recall"] > 0.4,
    )
    check(
        "the model with the highest accuracy is not the cheapest one",
        comparison["accuracy"].idxmax() != comparison["cost"].idxmin(),
    )
    check("KNN struggles on this rare class", comparison.loc["knn", "recall"] < 0.2)

PASS  comparison holds all four models
PASS  it reports accuracy, precision, recall, f1, missed, calls, and cost
PASS  logistic regression used your threshold, not the default 0.50
PASS  the model with the highest accuracy is not the cheapest one
PASS  KNN struggles on this rare class


### E4 · A fair comparison before the test set
The table above uses the test rows. For choosing between models you should look at the **training** rows first, the same way you chose `k` and `max_depth`. Here recall is the metric that matters, so cross-validate on recall.

In [27]:
cv_recall = None


# Write your code below. Your code must create the variable set to None above.

# 1. cv_recall: for every model in models, the average of
#    cross_val_score(model, X_train, y_train, cv=5, scoring="recall").
#    This uses the TRAINING rows only, so it is a fair comparison before the test set.
#    Hint: build a dictionary {name: score} or a small DataFrame
cv_recall = pd.Series(
    {
        name: round(
            cross_val_score(model, X_train, y_train, cv=5, scoring="recall").mean(), 3
        )
        for name, model, _ in models
    }
)

# 2. Display cv_recall next to the comparison table from E3.
display(cv_recall, comparison)

knn                    0.022
tree (balanced)        0.807
logistic regression    0.147
naive bayes            0.189
dtype: float64

,accuracy,precision,recall,f1,missed,calls,cost
model,,,,,,,
knn,0.92,0.000000,0.0000,0.000000,16,0,640
tree (balanced),0.81,0.270833,0.8125,0.406250,3,48,216
logistic regression,0.64,0.174419,0.9375,0.294118,1,86,212
naive bayes,0.91,0.333333,0.1250,0.181818,14,6,572


**Check E4**

In [28]:
if ready(cv_recall, "E4"):
    check("cv_recall covers all four models", len(cv_recall) == len(models))
    check(
        "it was computed on the training rows only",
        np.isclose(
            cv_recall["naive bayes"],
            round(
                cross_val_score(
                    naive_bayes, X_train, y_train, cv=5, scoring="recall"
                ).mean(),
                3,
            ),
        ),
    )
    check(
        "the tree has the highest cross-validated recall",
        cv_recall.idxmax() == "tree (balanced)",
    )
    check("KNN has the lowest", cv_recall.idxmin() == "knn")

PASS  cv_recall covers all four models
PASS  it was computed on the training rows only
PASS  the tree has the highest cross-validated recall
PASS  KNN has the lowest


**Write:** which model do you hand to the gym manager, and why? Use at least two numbers, and name one thing that would make you change your mind.

*Your answer:*
I would give the gym manager the tree (balanced) because it scored higher in every regard than the cheapest option and is only $4 more. 

# Part 5 · Two AI claims, and Assignment 2
An AI assistant gives this advice about the refund project:

> 1. "Use the default threshold of 0.5. That is the standard cut-off for classification."
> 2. "Try several classifiers on the test set and pick the one with the highest accuracy."

**Write:** answer both claims, using your own numbers from today.

*Your answer:*

1. False, the threshold of 0.5 catches 1/15 requests, compared to 0.05 which catches 15/16 requests

2. False, accuracy can be misleading when using imbalanced data. KNN had the highest accuracy but caught 0 refund requests. 

### Assignment 2 · what today prepares you for
The assignment gives you a new table, a new business question, and a cost for each kind of mistake. You will:
1. find the target and the features, and check for a column that would not exist at prediction time,
2. split the data and report the baseline,
3. build several classifiers,
4. tune one setting with cross-validation on the training rows,
5. choose a threshold using the costs,
6. compare with more than accuracy, and inspect the errors,
7. pick one model, defend it in writing, and critique an AI-generated analysis.

Everything on that list, you have now done at least once.

# Exit ticket
Answer without running more code.
1. A model gives a member a refund risk of 0.48. Does that mean "no"? Explain.
2. Our KNN model had the highest accuracy in the comparison table and the highest cost. How is that possible?
3. Name one thing Naive Bayes assumes that is not true about our features, and say why the model can still be useful.

*Your answers:*
1. 0.48 only means "no" if you are cutting at 0.50. This is because a 48% risk is very high and needs intervention. 
2. KNN ended up missing 16 of the 16 refund requests, which causes it to lose significantly more than models who correctly predicted refund.
3. Naive Bayes assumes that features are independent to one another. It only ranks relative risk rather than estimating many combined possibilities.

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Submit this notebook with its outputs.